# TradeSense · Component 03 · Protocol B2 · step 2: normalisation N0/N1 and the conditional autoencoder (AE-ctx)

What this notebook does (all under the frozen Protocol B2 stored in `layer3_v3/`):
1. checks that the protocol hash is intact and that Layer 3 v3 exists; **TEST-B / TEST-3 are never loaded** (Aug bars only up to 2024-08-31);
2. unit tests of the library (N1 causality, N1 = N0 for the first day, AE-ctx causality, reproducibility, OOF fold exclusion);
3. fits N0 on Layer 1 TRAIN, compares it with the v1 preprocessing files, and builds N1 (causal 7-day statistics);
4. trains AE-ctx on genuine Layer 1 TRAIN bars: 3 seeds on the full train (final model) and 5 time-block out-of-fold models (for XGBoost set B), for N0 and N1, BTC and ETH (resumable; weights saved);
5. scores genuine Layer 1, genuine Aug (to Aug 31), and every injected set replica (TRAIN-3, VAL-3, CAL-3); TRAIN-3 gets out-of-fold AE columns;
6. evaluates AE-ctx and the robust-z baseline under N0/N1 and thresholds T0/T1: DEV recall (VAL-3), drift stress (genuine CAL bars), CAL-3 recall; applies the pre-registered rules H-B1 and H-B2;
7. share block.

CPU runtime. The AE trainings are the long part (about 30 to 60 minutes in total); the notebook stops itself after `MAX_RUN_MINUTES` and continues on the next *Run all*.

In [ ]:
import os, json, glob, time, hashlib, warnings, datetime
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
import torch; torch.set_num_threads(max(1, os.cpu_count() or 1))
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'; (OUT/'ae').mkdir(parents=True, exist_ok=True); (OUT/'scores').mkdir(exist_ok=True)
SYMBOLS = ['BTCUSDT', 'ETHUSDT']; SEEDS = [11, 22, 33]; MODES = ['N0', 'N1']
MAX_RUN_MINUTES = float(os.environ.get('MAX_RUN_MINUTES', 150)); T_START = time.time()
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
def out_of_time(): return (time.time() - T_START) / 60 > MAX_RUN_MINUTES
# protocol freeze must exist and match the stored text
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'run notebook 27 first (it freezes the protocol)'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol was changed'
say('protocol B2 intact, sha256 ' + ph[:16])

Mounted at /content/drive
protocol B2 intact, sha256 443f99d5faf5929f


## 1 · Library

In [ ]:
open("b2_lib.py", "w").write(r'''
"""TradeSense C03 - Protocol B2 shared library: normalisation N0/N1, conditional autoencoder (AE-ctx), burden thresholds T0/T1, recall."""
import numpy as np, pandas as pd, torch, torch.nn as nn, time, json, hashlib
from numpy.lib.stride_tricks import sliding_window_view

FEATS = ['log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'rush_order_density', 'volume_acceleration', 'large_trade_count']
LOGC = [3, 5]; WIN7 = 10080; MINP = 1440; CTX = 15; WARM = 30; GAPM = 5; CLIP = 20.0
BUDGETS = [1, 2, 4]; PRIMARY = 2; QGRID = [0.99, 0.993, 0.995, 0.997, 0.998, 0.999, 0.9993, 0.9995, 0.9997, 0.9998, 0.9999, 0.99995]
FAMS = {1: 'pump_and_dump_like', 2: 'fake_breakout_like', 3: 'abnormal_volume_burst', 4: 'trade_flow_imbalance', 5: 'wash_like_aggregate_proxy',
        6: 'liquidity_stress_proxy', 7: 'momentum_ignition_like', 8: 'slow_accumulation_like'}


# ---------------------------------------------------------------- normalisation
def to_x(F):
    x = np.asarray(F[FEATS] if hasattr(F, 'columns') else F, dtype='float64').copy()
    for j in LOGC: x[:, j] = np.log1p(x[:, j])
    return x


def fit_n0(F_train):
    """median / (IQR/1.349) on log1p-transformed Layer 1 TRAIN bars (caller passes bars >= 15 only)."""
    x = to_x(F_train); med = np.median(x, axis=0); q75, q25 = np.percentile(x, [75, 25], axis=0); return {'median': med, 'scale': (q75 - q25) / 1.349}


def normalise(F, mode, n0):
    x = to_x(F)
    if mode == 'N0': med, scl = n0['median'][None, :], n0['scale'][None, :]
    else:
        df = pd.DataFrame(x); r = df.rolling(WIN7, min_periods=MINP)
        med = r.median().shift(1).to_numpy(); q75 = r.quantile(0.75).shift(1).to_numpy(); q25 = r.quantile(0.25).shift(1).to_numpy()
        scl = np.maximum((q75 - q25) / 1.349, 0.1 * n0['scale'][None, :])
        bad = np.isnan(med) | np.isnan(scl); med = np.where(bad, n0['median'][None, :], med); scl = np.where(bad, n0['scale'][None, :], scl)
    return np.clip((x - med) / scl, -CLIP, CLIP)


def context(z):
    """causal context: mean and std (ddof 1) of the previous CTX bars; rows < CTX are zero-filled and never used."""
    n = len(z); m = np.zeros_like(z); s = np.zeros_like(z)
    w = sliding_window_view(z, CTX, axis=0)[: n - CTX]            # window k = bars k..k+14 -> bar k+15
    m[CTX:] = w.mean(axis=2); s[CTX:] = w.std(axis=2, ddof=1); return m, s


# ---------------------------------------------------------------- conditional autoencoder
class AECtx(nn.Module):
    def __init__(self, d=6, b=3):
        super().__init__(); self.enc = nn.Sequential(nn.Linear(d + 2 * d, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, b))
        self.dec = nn.Sequential(nn.Linear(b + 2 * d, 32), nn.ReLU(), nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, d))

    def forward(self, z, cm, cs):
        c = torch.cat([cm, cs], 1); return self.dec(torch.cat([self.enc(torch.cat([z, c], 1)), c], 1))


def train_ae(z, idx, seed, max_epochs=150, patience=10, batch=512, log=None):
    """z normalised genuine series; idx = bar indices allowed for fitting (valid, inside Layer 1 TRAIN). 90% fit / 10% early stop, 15-bar gap."""
    torch.manual_seed(seed); rng = np.random.default_rng(seed); cm, cs = context(z)
    idx = np.sort(idx); cut = int(0.9 * len(idx)); fit = idx[:cut]; val = idx[idx > idx[cut] + CTX]
    T = lambda a: torch.tensor(a, dtype=torch.float32)
    Zf, Mf, Sf = T(z[fit]), T(cm[fit]), T(cs[fit]); Zv, Mv, Sv = T(z[val]), T(cm[val]), T(cs[val])
    net = AECtx(); opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5); best, bad, state = 1e18, 0, None
    for ep in range(max_epochs):
        net.train(); perm = torch.tensor(rng.permutation(len(fit)))
        for i in range(0, len(perm), batch):
            b = perm[i:i + batch]; opt.zero_grad(); loss = ((net(Zf[b], Mf[b], Sf[b]) - Zf[b]) ** 2).mean(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): v = float(((net(Zv, Mv, Sv) - Zv) ** 2).mean())
        if v < best - 1e-6: best, bad, state = v, 0, {k: x.clone() for k, x in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(state); net.eval(); return net, {'epochs': ep + 1, 'best_val': best, 'fit_rows': int(len(fit)), 'val_rows': int(len(val)), 'fit_min': int(fit.min()), 'fit_max': int(fit.max())}


def ae_errors(net, z):
    cm, cs = context(z); T = lambda a: torch.tensor(a, dtype=torch.float32)
    with torch.no_grad(): rec = net(T(z), T(cm), T(cs)).numpy()
    return (z - rec) ** 2


def p99_scale(err, idx): return np.percentile(err[idx], 99, axis=0) + 1e-12


def ae_columns(err, scale):
    """8 columns: fmax score, fmean score, 6 per-feature errors / TRAIN p99 scale."""
    r = err / scale[None, :]; return np.column_stack([r.max(axis=1), err.mean(axis=1), r])


# ---------------------------------------------------------------- burden: episodes, thresholds, recall
def episodes(alarm, gap=GAPM):
    a = np.flatnonzero(alarm)
    if len(a) == 0: return 0
    return int(1 + np.sum(np.diff(a) > gap))


def eps_per_day(alarm, valid):
    a = alarm & valid; days = valid.sum() / 1440.0
    return episodes(a) / max(days, 1e-9)


def valid_mask(n, lo, hi): m = np.zeros(n, bool); m[max(lo, WARM):hi] = True; return m


def t0_threshold(score, valid, budget):
    """lowest quantile-grid threshold (descending probability grid, lowering the threshold step by step) with episodes/day <= budget."""
    s = score[valid]; best = None
    for q in sorted(QGRID, reverse=True):
        thr = np.quantile(s, q)
        if eps_per_day(score > thr, valid) <= budget: best = thr
        else: break
    return best if best is not None else np.quantile(s, max(QGRID))


def rolling_thr(score, q):
    return pd.Series(score).rolling(WIN7, min_periods=MINP).quantile(q).shift(1).to_numpy()


def t1_q(score, valid, budget):
    """smallest quantile level q (lowest rolling threshold) with episodes/day <= budget on the genuine DEV bars."""
    for q in QGRID:
        thr = rolling_thr(score, q); alarm = (score > np.where(np.isnan(thr), np.inf, thr))
        if eps_per_day(alarm, valid) <= budget: return q
    return QGRID[-1]


def alarms(score, rule, par):
    if rule == 'T0': return score > par
    thr = rolling_thr(score, par); return score > np.where(np.isnan(thr), np.inf, thr)


def recall_by_family(alarm, ev, seg_ok=None):
    """ev rows: t0, dur, family, (replica); detected if an alarm falls in [t0, t0+dur-1]."""
    c = np.concatenate([[0], np.cumsum(alarm.astype(np.int64))]); out = {}
    for fam, sub in ev.groupby('family'):
        hit = [(c[r.t0 + r.dur] - c[r.t0]) > 0 for r in sub.itertuples()]; out[int(fam)] = (int(np.sum(hit)), int(len(hit)))
    return out


def bootstrap_ci(hits, n=1000, seed=0):
    rng = np.random.default_rng(seed); h = np.asarray(hits, float)
    if len(h) == 0: return (np.nan, np.nan)
    b = [h[rng.integers(0, len(h), len(h))].mean() for _ in range(n)]; return (float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5)))


def macro(rec, fams):
    v = [rec[f][0] / rec[f][1] for f in fams if f in rec and rec[f][1] > 0]; return float(np.mean(v)) if v else float('nan')


def sha_bytes(b): return hashlib.sha256(b).hexdigest()

''')

import sys, importlib; sys.path.insert(0, '.'); import b2_lib as B; importlib.reload(B); (OUT/'b2_lib.py').write_text(Path('b2_lib.py').read_text())
say('library sha256 ' + sha_file('b2_lib.py'))

library sha256 1b0b46630763c944


## 2 · Unit tests

In [ ]:
rng = np.random.default_rng(0); n = 20000
Ft = pd.DataFrame({'log_returns': rng.standard_t(4, n) * 3e-4, 'trade_flow_imbalance': rng.uniform(-1, 1, n), 'hl_spread_ratio': rng.uniform(1e-4, 1e-3, n),
                   'rush_order_density': np.exp(rng.normal(4, .5, n)), 'volume_acceleration': rng.normal(0, 1, n), 'large_trade_count': rng.poisson(20, n).astype(float)})
n0t = B.fit_n0(Ft.iloc[15:15000]); z0 = B.normalise(Ft, 'N0', n0t); z1 = B.normalise(Ft, 'N1', n0t); res = {}
res['T01 N1 equals N0 until the minimum history (1440 bars)'] = bool(np.array_equal(z0[:1440], z1[:1440]))
F2 = Ft.copy(); F2.iloc[12000:, 3] *= 5; res['T02 N1 is causal (changing bars after t leaves values <= t unchanged)'] = bool(np.array_equal(B.normalise(F2, 'N1', n0t)[:12000], z1[:12000]))
x = B.to_x(Ft); res['T03 N0 statistics equal an independent recompute'] = bool(np.allclose(n0t['median'], np.median(x[15:15000], axis=0)) and np.allclose(n0t['scale'] * 1.349, np.subtract(*np.percentile(x[15:15000], [75, 25], axis=0))))
t = 17000; w = z1[t - 15:t]; cm, cs = B.context(z1); res['T04 context equals a plain recompute (previous 15 bars, ddof 1)'] = bool(np.allclose(cm[t], w.mean(0)) and np.allclose(cs[t], w.std(0, ddof=1)))
net, info = B.train_ae(z0, np.arange(30, 6000), 11, max_epochs=3); e1 = B.ae_errors(net, z0); z0b = z0.copy(); z0b[9000:] *= 3
res['T05 AE-ctx scores are causal'] = bool(np.array_equal(e1[:9000], B.ae_errors(net, z0b)[:9000]))
net2, info2 = B.train_ae(z0, np.arange(30, 6000), 11, max_epochs=3); res['T06 same seed reproduces the AE'] = bool(np.allclose(e1, B.ae_errors(net2, z0), atol=1e-6))
res['T07 AE fit rows inside the allowed training range'] = bool(info['fit_min'] >= 30 and info['fit_max'] < 6000)
sc = np.abs(rng.normal(size=n)); sc[rng.integers(0, n, 40)] += 8; valid = B.valid_mask(n, 0, n)
thr = B.t0_threshold(sc, valid, 2); alarm = sc > thr; plain = 0; prev = -99
for i in np.flatnonzero(alarm & valid):
    if i - prev > B.GAPM: plain += 1
    prev = i
res['T08 episode counter equals a plain loop; T0 threshold respects the budget'] = bool(plain == B.episodes(alarm & valid) and B.eps_per_day(alarm, valid) <= 2 + 1e-9)
ev = pd.DataFrame({'t0': [100, 500], 'dur': [10, 5], 'family': [1, 1]}); al = np.zeros(n, bool); al[504] = True
res['T09 recall rule: alarm inside [t0, t0+dur-1]'] = bool(B.recall_by_family(al, ev)[1] == (1, 2))
for k, v in res.items(): say(('PASS ' if v else 'FAIL ') + k)
assert all(res.values())

PASS T01 N1 equals N0 until the minimum history (1440 bars)
PASS T02 N1 is causal (changing bars after t leaves values <= t unchanged)
PASS T03 N0 statistics equal an independent recompute
PASS T04 context equals a plain recompute (previous 15 bars, ddof 1)
PASS T05 AE-ctx scores are causal
PASS T06 same seed reproduces the AE
PASS T07 AE fit rows inside the allowed training range
PASS T08 episode counter equals a plain loop; T0 threshold respects the budget
PASS T09 recall rule: alarm inside [t0, t0+dur-1]


## 3 · Data (no test data) and N0

In [ ]:
def find1(pat):
    c = [x for x in glob.glob(str(BASE/'**'/pat), recursive=True) if 'backup' not in x.lower()]; assert c, pat; return sorted(c)[0]
D = {}
for s in SYMBOLS:
    a = pd.read_parquet(find1(f'{s.lower()}_l1_feature_matrix_90d.parquet')); a = a.set_index('datetime') if 'datetime' in a.columns else a
    b = pd.read_parquet(BASE/'layer1_aug_oct_v1'/f'{s.lower()}_l1_feature_matrix_aug_oct_2024.parquet').iloc[:31*1440]      # genuine Aug only (to 2024-08-31): TEST-B is never loaded
    for d in (a, b): d.loc[d.index[:15], 'volume_acceleration'] = 0.0
    sets = {}
    for name in ['TRAIN', 'VAL', 'CAL']:
        f = pd.read_parquet(L3/f'l3v3_{s}_{name}.parquet'); ev = pd.read_csv(L3/f'l3v3_{s}_{name}_events.csv'); sets[name] = {}
        for r in sorted(f.replica.unique()): sets[name][int(r)] = (f[f.replica == r].reset_index(drop=True), ev[ev.replica == r].reset_index(drop=True))
    D[s] = {'GEN_L1': a[B.FEATS].reset_index(drop=True), 'GEN_AO': b[B.FEATS].reset_index(drop=True), 'sets': sets}
    say(f'{s}: genuine Layer 1 {len(a)} bars, genuine Aug {len(b)} bars (2024-08-01..08-31); sets ' + str({k: len(v) for k, v in sets.items()}) + ' replicas')
assert not any('TEST' in str(k) for s in SYMBOLS for k in D[s]['sets'])
N0 = {s: B.fit_n0(D[s]['GEN_L1'].iloc[15:90720]) for s in SYMBOLS}
for s in SYMBOLS:
    pj = glob.glob(str(BASE/'**'/'models_autoencoder_v1'/'preprocessing'/f'{"btc" if s == "BTCUSDT" else "eth"}_preprocessing.json'), recursive=True)
    if pj:
        j = json.load(open(pj[0])); dm = np.max(np.abs(np.array(j['median']) - N0[s]['median'])); ds = np.max(np.abs(np.array(j['scale']) - N0[s]['scale']) / np.array(j['scale']))
        say(f'{s}: N0 versus the v1 preprocessing file: max |median diff| {dm:.2e}, max relative scale diff {ds:.2e}'); assert dm < 1e-6 and ds < 1e-6
    else: say(f'{s}: v1 preprocessing file not found (check skipped)')

BTCUSDT: genuine Layer 1 129600 bars, genuine Aug 44640 bars (2024-08-01..08-31); sets {'TRAIN': 2, 'VAL': 2, 'CAL': 1} replicas
ETHUSDT: genuine Layer 1 129600 bars, genuine Aug 44640 bars (2024-08-01..08-31); sets {'TRAIN': 2, 'VAL': 2, 'CAL': 1} replicas
BTCUSDT: N0 versus the v1 preprocessing file: max |median diff| 0.00e+00, max relative scale diff 0.00e+00
ETHUSDT: N0 versus the v1 preprocessing file: max |median diff| 4.44e-16, max relative scale diff 4.47e-16


## 4 · Train AE-ctx (final seeds and out-of-fold models)

In [ ]:
TRAIN_HI = 90720; E = np.linspace(B.WARM, TRAIN_HI, 6).astype(int); PURGE = 60
def folds(): return [(E[k], E[k + 1]) for k in range(5)]
Z = {}; FIT_LOG = {}
for s in SYMBOLS:
    for mode in MODES:
        Z[(s, mode, 'GEN_L1')] = B.normalise(D[s]['GEN_L1'], mode, N0[s]); Z[(s, mode, 'GEN_AO')] = B.normalise(D[s]['GEN_AO'], mode, N0[s])
def path(s, mode, tag): return OUT/'ae'/f'{s}_{mode}_{tag}.pt'
import torch
for s in SYMBOLS:
    for mode in MODES:
        z = Z[(s, mode, 'GEN_L1')]; allidx = np.arange(B.WARM, TRAIN_HI)
        jobs = [(f'final_s{sd}', allidx, sd) for sd in SEEDS] + [(f'fold{k}', allidx[(allidx < a - PURGE) | (allidx >= b + PURGE)], 11) for k, (a, b) in enumerate(folds())]
        for tag, idx, sd in jobs:
            p = path(s, mode, tag)
            if p.exists(): continue
            if out_of_time(): break
            t = time.time(); net, info = B.train_ae(z, idx, sd); torch.save({'state': net.state_dict(), 'info': info}, p)
            FIT_LOG[(s, mode, tag)] = info; say(f'{s} {mode} {tag}: epochs {info["epochs"]}, val MSE {info["best_val"]:.4f}, fit rows {info["fit_rows"]} ({time.time()-t:.0f}s)')
done = sum(path(s, m, t).exists() for s in SYMBOLS for m in MODES for t in [f'final_s{x}' for x in SEEDS] + [f'fold{k}' for k in range(5)])
say(f'AE models on disk: {done} of 32'); INCOMPLETE = done < 32
if INCOMPLETE: say('>>> time budget reached: Run all again to continue (finished models are kept). The remaining cells are skipped until all 32 exist. <<<')

BTCUSDT N0 final_s11: epochs 119, val MSE 0.0397, fit rows 81621 (80s)
BTCUSDT N0 final_s22: epochs 120, val MSE 0.0405, fit rows 81621 (86s)
BTCUSDT N0 final_s33: epochs 50, val MSE 0.0442, fit rows 81621 (39s)
BTCUSDT N0 fold0: epochs 83, val MSE 0.0429, fit rows 65242 (48s)
BTCUSDT N0 fold1: epochs 150, val MSE 0.0405, fit rows 65188 (88s)
BTCUSDT N0 fold2: epochs 137, val MSE 0.0410, fit rows 65188 (81s)
BTCUSDT N0 fold3: epochs 126, val MSE 0.0421, fit rows 65188 (72s)
BTCUSDT N0 fold4: epochs 128, val MSE 0.0522, fit rows 65242 (76s)
BTCUSDT N1 final_s11: epochs 99, val MSE 0.0447, fit rows 81621 (71s)
BTCUSDT N1 final_s22: epochs 56, val MSE 0.0433, fit rows 81621 (41s)
BTCUSDT N1 final_s33: epochs 61, val MSE 0.0435, fit rows 81621 (47s)
BTCUSDT N1 fold0: epochs 94, val MSE 0.0416, fit rows 65242 (47s)
BTCUSDT N1 fold1: epochs 112, val MSE 0.0443, fit rows 65188 (57s)
BTCUSDT N1 fold2: epochs 96, val MSE 0.0448, fit rows 65188 (49s)
BTCUSDT N1 fold3: epochs 140, val MSE 0.0448,

## 5 · Score every series

In [ ]:
def load_net(s, mode, tag):
    ck = torch.load(path(s, mode, tag)); net = B.AECtx(); net.load_state_dict(ck['state']); net.eval(); return net
SC = {}
if not INCOMPLETE:
    # fold exclusion check (OOF): the fold-k AE saw no bar in [E_k - purge, E_k+1 + purge)
    for s in SYMBOLS:
        for mode in MODES:
            for k, (a, b) in enumerate(folds()):
                info = torch.load(path(s, mode, f'fold{k}'))['info']; assert info['fit_rows'] > 0
    say('PASS T10 all 5 out-of-fold models exist per symbol and normalisation; each was fitted on bars outside its block and a 60-bar purge (by construction of the index list)')
    for s in SYMBOLS:
        for mode in MODES:
            fin = [load_net(s, mode, f'final_s{sd}') for sd in SEEDS]; fol = [load_net(s, mode, f'fold{k}') for k in range(5)]
            zg = Z[(s, mode, 'GEN_L1')]; err_g = np.mean([B.ae_errors(n_, zg) for n_ in fin], axis=0); scale = B.p99_scale(err_g, np.arange(B.WARM, TRAIN_HI))
            def cols_for(z, oof=False):
                err = np.mean([B.ae_errors(n_, z) for n_ in fin], axis=0)
                if oof:
                    for k, (a, b) in enumerate(folds()): err[a:b] = B.ae_errors(fol[k], z)[a:b]
                return B.ae_columns(err, scale).astype('float32')
            SC[(s, mode, 'GEN_L1')] = cols_for(zg); SC[(s, mode, 'GEN_AO')] = cols_for(Z[(s, mode, 'GEN_AO')])
            for name, reps in D[s]['sets'].items():
                for r, (f, ev) in reps.items():
                    z = B.normalise(f, mode, N0[s]); Z[(s, mode, f'{name}_r{r}')] = z; SC[(s, mode, f'{name}_r{r}')] = cols_for(z, oof=(name == 'TRAIN'))
            np.savez_compressed(OUT/'scores'/f'ae_ctx_{s}_{mode}.npz', **{k[2]: v for k, v in SC.items() if k[0] == s and k[1] == mode}, scale=scale)
            say(f'{s} {mode}: scored {sum(1 for k in SC if k[0]==s and k[1]==mode)} series; AE error scale (train p99) ' + str(np.round(scale, 3)))

PASS T10 all 5 out-of-fold models exist per symbol and normalisation; each was fitted on bars outside its block and a 60-bar purge (by construction of the index list)
BTCUSDT N0: scored 7 series; AE error scale (train p99) [0.225 0.583 0.258 0.567 0.319 0.263]
BTCUSDT N1: scored 7 series; AE error scale (train p99) [0.282 0.613 0.29  0.656 0.349 0.287]
ETHUSDT N0: scored 7 series; AE error scale (train p99) [0.265 0.565 0.243 0.637 0.375 0.409]
ETHUSDT N1: scored 7 series; AE error scale (train p99) [0.298 0.577 0.255 0.649 0.411 0.42 ]


## 6 · Evaluation and pre-registered rules

In [ ]:
RES = {}
if not INCOMPLETE:
    def rz_cols(z): return np.abs(z).max(axis=1)
    def models(s, mode, key):
        c = SC[(s, mode, key)]; z = Z[(s, mode, key)]
        return {'AE-ctx fmax': c[:, 0].astype('float64'), 'AE-ctx fmean': c[:, 1].astype('float64'), 'robust-z': rz_cols(z)}
    L1V = B.valid_mask(129600, 90720, 129600); CALV = B.valid_mask(31*1440, 1440, 31*1440)
    for s in SYMBOLS:
        for mode in MODES:
            gl, ga = models(s, mode, 'GEN_L1'), models(s, mode, 'GEN_AO')
            for mname in gl:
                for rule in ['T0', 'T1']:
                    for bud in B.BUDGETS:
                        par = B.t0_threshold(gl[mname], L1V, bud) if rule == 'T0' else B.t1_q(gl[mname], L1V, bud)
                        dev_eps = B.eps_per_day(B.alarms(gl[mname], rule, par), L1V); cal_eps = B.eps_per_day(B.alarms(ga[mname], rule, par), CALV)
                        rec = {}
                        for name in ['VAL', 'CAL']:
                            tot = {}
                            for r, (f, ev) in D[s]['sets'][name].items():
                                sc_ = models(s, mode, f'{name}_r{r}')[mname]; al = B.alarms(sc_, rule, par)
                                for fam, (h, n_) in B.recall_by_family(al, ev).items(): a_, b_ = tot.get(fam, (0, 0)); tot[fam] = (a_ + h, b_ + n_)
                            rec[name] = tot
                        RES[(s, mode, mname, rule, bud)] = {'dev_eps': dev_eps, 'cal_eps': cal_eps, 'rec': rec}
    def agg(mode, mname, rule, bud, name, fams):
        v = [B.macro(RES[(s, mode, mname, rule, bud)]['rec'][name], fams) for s in SYMBOLS]; return float(np.mean(v))
    OLD, NEW = [1, 2, 3, 4, 5, 6], [7, 8]; rows = []
    for mname in ['AE-ctx fmax', 'AE-ctx fmean', 'robust-z']:
        for mode in MODES:
            for rule in ['T0', 'T1']:
                for bud in B.BUDGETS:
                    rows.append({'model': mname, 'norm': mode, 'thr': rule, 'budget/day': bud,
                                 'DEV genuine eps/day': np.mean([RES[(s, mode, mname, rule, bud)]['dev_eps'] for s in SYMBOLS]),
                                 'CAL genuine eps/day': np.mean([RES[(s, mode, mname, rule, bud)]['cal_eps'] for s in SYMBOLS]),
                                 'VAL-3 recall 6 old': agg(mode, mname, rule, bud, 'VAL', OLD), 'VAL-3 recall 2 new': agg(mode, mname, rule, bud, 'VAL', NEW),
                                 'CAL-3 recall 8': agg(mode, mname, rule, bud, 'CAL', OLD + NEW)})
    TAB = pd.DataFrame(rows); say(TAB.round(3).to_string(index=False))
    TAB.to_csv(OUT/'ae_ctx_eval_table.csv', index=False)
    say(''); say('--- pre-registered rules (primary budget 2 episodes/day) ---')
    DEC = {}
    for mname in ['AE-ctx fmax', 'AE-ctx fmean', 'robust-z']:
        i0 = np.mean([RES[(s, 'N0', mname, 'T0', 2)]['cal_eps'] for s in SYMBOLS]) / 2.0; i1 = np.mean([RES[(s, 'N1', mname, 'T1', 2)]['cal_eps'] for s in SYMBOLS]) / 2.0
        ex0, ex1 = max(i0 - 1, 0), max(i1 - 1, 0); hb1_premise = i0 > 2.0; hb1 = (ex1 <= 0.5 * ex0) and (0.5 <= i1 <= 2.0)
        r0 = agg('N0', mname, 'T1', 2, 'VAL', OLD + NEW); r1 = agg('N1', mname, 'T1', 2, 'VAL', OLD + NEW); hb2 = r1 >= r0 - 0.05
        DEC[mname] = {'inflation_N0_T0': i0, 'inflation_N1_T1': i1, 'H-B1 premise (N0/T0 alarm rate > 2x budget)': bool(hb1_premise), 'H-B1 (N1/T1 cuts excess >= 50% and within 0.5..2x)': bool(hb1),
                      'VAL-3 recall N0 T1': r0, 'VAL-3 recall N1 T1': r1, 'H-B2 (non-inferior within 0.05)': bool(hb2), 'adopt N1': bool(hb1 and hb2)}
        say(f'{mname}: CAL alarm-rate inflation (x budget) N0/T0 {i0:.2f} -> N1/T1 {i1:.2f}; H-B1 premise {hb1_premise}, H-B1 {hb1}; VAL-3 macro recall (8 families, T1) N0 {r0:.3f} vs N1 {r1:.3f}, H-B2 {hb2}; adopt N1: {hb1 and hb2}')
    (OUT/'ae_ctx_decisions.json').write_text(json.dumps(DEC, indent=1))

       model norm thr  budget/day  DEV genuine eps/day  CAL genuine eps/day  VAL-3 recall 6 old  VAL-3 recall 2 new  CAL-3 recall 8
 AE-ctx fmax   N0  T0           1                0.944                0.883               0.186               0.125           0.140
 AE-ctx fmax   N0  T0           2                1.296                1.283               0.206               0.146           0.170
 AE-ctx fmax   N0  T0           4                3.852                4.017               0.331               0.292           0.322
 AE-ctx fmax   N0  T1           1                0.815                1.050               0.112               0.062           0.098
 AE-ctx fmax   N0  T1           2                1.389                1.583               0.145               0.083           0.115
 AE-ctx fmax   N0  T1           4                3.167                3.500               0.243               0.208           0.210
 AE-ctx fmax   N1  T0           1                0.815                0.450 

## 7 · Share block

In [ ]:
block = '\n'.join(LINES); (OUT/'b2_step2_share_block.txt').write_text(block)
print('\n===== SHARE BLOCK (copy everything below, or attach models_b2_v1/b2_step2_share_block.txt) =====\n' + block[-9000:])


===== SHARE BLOCK (copy everything below, or attach models_b2_v1/b2_step2_share_block.txt) =====
ddof 1)
PASS T05 AE-ctx scores are causal
PASS T06 same seed reproduces the AE
PASS T07 AE fit rows inside the allowed training range
PASS T08 episode counter equals a plain loop; T0 threshold respects the budget
PASS T09 recall rule: alarm inside [t0, t0+dur-1]
BTCUSDT: genuine Layer 1 129600 bars, genuine Aug 44640 bars (2024-08-01..08-31); sets {'TRAIN': 2, 'VAL': 2, 'CAL': 1} replicas
ETHUSDT: genuine Layer 1 129600 bars, genuine Aug 44640 bars (2024-08-01..08-31); sets {'TRAIN': 2, 'VAL': 2, 'CAL': 1} replicas
BTCUSDT: N0 versus the v1 preprocessing file: max |median diff| 0.00e+00, max relative scale diff 0.00e+00
ETHUSDT: N0 versus the v1 preprocessing file: max |median diff| 4.44e-16, max relative scale diff 4.47e-16
BTCUSDT N0 final_s11: epochs 119, val MSE 0.0397, fit rows 81621 (80s)
BTCUSDT N0 final_s22: epochs 120, val MSE 0.0405, fit rows 81621 (86s)
BTCUSDT N0 final_s33: epo